# Exercise 2.3.10 — rewrite `get_actor_and_critic`

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `2.3 PPO`  
**Notebook:** `2.3_PPO_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=2.3.10](https://delta-drills.vercel.app/?arena_exercise=2.3.10)


# [2.3] - PPO (exercises)

> **ARENA [Streamlit Page](https://arena-chapter2-rl.streamlit.app/03_[2.3]_PPO)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part3_ppo/2.3_PPO_exercises.ipynb?t=20260303) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part3_ppo/2.3_PPO_solutions.ipynb?t=20260303)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-23.png" width="350">

# Introduction

Proximal Policy Optimization (PPO) is a cutting-edge reinforcement learning algorithm that has gained significant attention in recent years. As an improvement over traditional policy optimization methods, PPO addresses key challenges such as sample efficiency, stability, and robustness in training deep neural networks for reinforcement learning tasks. With its ability to strike a balance between exploration and exploitation, PPO has demonstrated remarkable performance across a wide range of complex environments, including robotics, game playing, and autonomous control systems.

In this section, you'll build your own agent to perform PPO on the CartPole environment. By the end, you should be able to train your agent to near perfect performance in about 30 seconds. You'll also be able to try out other things like **reward shaping**, to make it easier for your agent to learn to balance, or to do fun tricks! There are also additional exercises which allow you to experiment with other tasks, including **Atari** and the 3D physics engine **MuJoCo**.

A lot of the setup as we go through these exercises will be similar to what we did yesterday for DQN, so you might find yourself moving quickly through certain sections.

For a lecture on the material today, which provides some high-level understanding before you dive into the material, watch the video below:

<iframe width="540" height="304" src="https://www.youtube.com/embed/NyV1eb0vWLA" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

## Content & Learning Objectives

### 0️⃣ Whirlwind Tour of PPO

In this non-exercise-based section, we discuss some of the mathematical intuitions underpinning PPO. It's not compulsory to go through all of it (and various recommended reading material / online lectures may provide better alternatives), although we strongly recommend everyone to at least read the summary boxes at the end of each subsection.

> ##### Learning Objectives
>
> - Understand the mathematical intuitions of PPO
> - Learn how expressions like the PPO objective function are derived

### 1️⃣ Setting up our agent

We'll start by building up most of our PPO infrastructure. Most importantly, this involves creating our actor & critic networks and writing methods using both of them which take steps in our environment. The result will be a `PPOAgent` and `ReplayMemory` class, analogous to our `DQNAgent` and `ReplayBuffer` from yesterday.

> ##### Learning Objectives
>
> - Understand the difference between the actor & critic networks, and what their roles are
> - Learn about & implement generalised advantage estimation
> - Build a replay memory to store & sample experiences
> - Design an agent class to step through the environment & record experiences

### 2️⃣ Learning Phase

The PPO objective function is considerably more complex than DQN and involves a lot of moving parts. In this section we'll go through each of those parts one by one, understanding its role and how to implement it.

> ##### Learning Objectives
>
> - Implement the total objective function (sum of three separate terms)
> - Understand the importance of each of these terms for the overall algorithm
> - Write a function to return an optimizer and learning rate scheduler for your model

### 3️⃣ Training Loop

Lastly, we'll assemble everything together into a `PPOTrainer` class just like our `DQNTrainer` class from yesterday, and use it to train on CartPole. We can also go further than yesterday by using **reward shaping** to fast-track our agent's learning trajectory.

> ##### Learning Objectives
>
> - Build a full training loop for the PPO algorithm
> - Train our agent, and visualise its performance with Weights & Biases media logger
> - Use reward shaping to improve your agent's training (and make it do tricks!)

### 4️⃣ Atari

Now that we've got training working on CartPole, we'll extend to the more complex environment of Atari. There are no massively new concepts in this section, although we do have to deal with a very different architecture that takes into account the visual structure of our observations (Atari frames), in particular this will also require a shared architecture between the actor & critic networks.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/atari-demo.gif" width="200"><br>

> ##### Learning Objectives
>
> - Understand how PPO can be used in visual domains, with appropriate architectures (CNNs)
> - Understand the idea of policy and value heads
> - Train an agent to solve the Breakout environment

### 5️⃣ MuJoCo

The last new set of environments we'll look at is MuJoCo. This is a 3D physics engine, which you might be familiar with in the context of OpenAI's famous backflipping noodle which laid the background for RLHF (see tomorrow for more on this!). The most important concept MuJoCo introduces for us is the idea of a **continuous action space**, where actions aren't chosen discretely from a set of finite options but are sampled from some probability distribution (in this case, a parameterized normal distribution). This is one setting that PPO can work in, but DQN can't.

<img src="https://images.ctfassets.net/kftzwdyauwt9/cf6fdf49-ea9e-489d-eb53eceeebc7/03dec4ea90925c03dea2ee6c4976e921/humanfeedbackjump.gif?w=2048&q=90&fm=webp" width="200"><br>

> ##### Learning Objectives
>
> - Understand how PPO can be used to train agents in continuous action spaces
> - Install and interact with the MuJoCo physics engine
> - Train an agent to solve the Hopper environment

### ☆ Bonus

We conclude with a set of optional bonus exercises, which you can try out before moving on to the RLHF sections.

## Notes on today's workflow

Your implementation might get good benchmark scores by the end of the day, but don't worry if it struggles to learn the simplest of tasks. RL can be frustrating because the feedback you get is extremely noisy: the agent can fail even with correct code, and succeed with buggy code. Forming a systematic process for coping with the confusion and uncertainty is the point of today, more so than producing a working PPO implementation.

Some parts of your process could include:

- Forming hypotheses about why it isn't working, and thinking about what tests you could write, or where you could set a breakpoint to confirm the hypothesis.
- Implementing some of the even more basic gymnasium environments and testing your agent on those.
- Getting a sense for the meaning of various logged metrics, and what this implies about the training process
- Noticing confusion and sections that don't make sense, and investigating this instead of hand-waving over it.

## Readings

In section 0️⃣, we've included a whirlwind tour of PPO which is specifically tailored to today's exercises. Going through the entire thing isn't required (since it can get quite mathematically dense), but **we strongly recommend everyone at least read the summary boxes at the end of each subsection**. Many of the resources listed below are also useful, but they don't cover everything which is specifically relevant to today's exercises.

If you find this section sufficient then you can move on to the exercises, if not then other strongly recommended reading includes:

- [An introduction to Policy Gradient methods - Deep RL](https://www.youtube.com/watch?v=5P7I-xPq8u8) (20 mins)
    - This is a useful video which motivates the core setup of PPO (and in particular the clipped objective function) without spending too much time with the precise derivations. We recommend watching this video before doing the exercises.
    - Note - you can ignore the short section on multi-GPU setup.
    - Also, near the end the video says that PPO outputs parameters $\mu$ and $\sigma$ from which actions are sampled, this is true for non-discrete action spaces (which we'll be using later on) but we'll start by implementing PPO on CartPole meaning our observation and action space is discrete just like yesterday.
- [The 37 Implementation Details of Proximal Policy Optimization](https://iclr-blog-track.github.io/2022/03/25/ppo-implementation-details/#solving-pong-in-5-minutes-with-ppo--envpool)
    - This is not required reading before the exercises, but **it will be a useful reference point as you go through the exercises*- (and it's also a useful thing to take away from the course as a whole, since your future work in RL will likely be less guided than these exercises).
    - The good news is that you won't need all 37 of these today, so no need to read to the end.
    - We will be tackling the 13 "core" details, not in the same order as presented here. Some of the sections below are labelled with the number they correspond to in this page (e.g. **Minibatch Update ([detail #6](https://iclr-blog-track.github.io/2022/03/25/ppo-implementation-details/#:~:text=Mini%2Dbatch%20Updates))**).
- [Proximal Policy Optimization Algorithms](https://arxiv.org/pdf/1707.06347.pdf)
    - **This is not required reading before the exercises**, but it will be a useful reference point for many of the key equations as you go through the exercises. In particular, you will find up to page 5 useful.


### Optional Reading

- [Spinning Up in Deep RL - Vanilla Policy Gradient](https://spinningup.openai.com/en/latest/algorithms/vpg.html#background)
    - PPO is a fancier version of vanilla policy gradient, so if you're struggling to understand PPO it may help to look at the simpler setting first.
- [Spinning Up in Deep RL - PPO](https://spinningup.openai.com/en/latest/algorithms/ppo.html)
    - You don't need to follow all the derivations here, although as a general idea by the end you should at least have a qualitative understanding of what all the symbols represent.
- [Andy Jones - Debugging RL, Without the Agonizing Pain](https://andyljones.com/posts/rl-debugging.html)
    - You've already read this previously but it will come in handy again.
    - You'll want to reuse your probe environments from yesterday, or you can import them from the solution if you didn't implement them all.
- [Tricks from Deep RL Bootcamp at UC Berkeley](https://github.com/williamFalcon/DeepRLHacks/blob/master/README.md)
    - This contains more debugging tips that may be of use.
- [Lilian Weng Blog on PPO](https://lilianweng.github.io/posts/2018-04-08-policy-gradient/#ppo)
    - Her writing on ML topics is consistently informative and informationally dense.

## Setup code

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter2_rl"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import jaxtyping
except:
    %pip install wandb==0.18.7 einops gymnasium[atari,accept-rom-license,other,mujoco-py]==0.29.0 pygame jaxtyping

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import itertools
import os
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import einops
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch as t
import torch.nn as nn
import torch.optim as optim
import wandb
from IPython.display import HTML, display
from jaxtyping import Bool, Float, Int
from matplotlib.animation import FuncAnimation
from numpy.random import Generator
from torch import Tensor
from torch.distributions.categorical import Categorical
from torch.optim.optimizer import Optimizer
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part3_ppo"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part3_ppo.tests as tests
from part1_intro_to_rl.utils import set_global_seeds
from part3_ppo.utils import arg_help
from part21_dqn.solutions import (
    Probe1,
    Probe2,
    Probe3,
    Probe4,
    Probe5,
    get_episode_data_from_infos,
)
from plotly_utils import plot_cartpole_obs_and_dones
from rl_utils import make_env, prepare_atari_env

# Register our probes from last time
for idx, probe in enumerate([Probe1, Probe2, Probe3, Probe4, Probe5]):
    gym.envs.registration.register(id=f"Probe{idx + 1}-v0", entry_point=probe)

Arr = np.ndarray

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# 0️⃣ Whirlwind tour of PPO

> ##### Learning Objectives
>
> - Understand the mathematical intuitions of PPO
> - Learn how expressions like the PPO objective function are derived

This section is quite mathematically dense, and you'll cover a lot of it again as you go through the exercises (plus, understanding all the details behind PPO isn't strictly necessary to get all the benefits from this chapter). 

At the end of each subsection we've included a box which summarizes the main points covered so far which should help distill out the main ideas as you're reading, as well as a couple of questions which help you check your understanding. We strongly recommend reading at least the contents of these boxes and attempting the questions, and also reading the section at the end describing today's setup.

Also, an important note - to simplify notation, everything here assumes a finite-horizon setting and no discount factor i.e. $\gamma = 1$. If we remove these assumptions, not a whole lot changes, but we wanted to higlight it to avoid confusion.

## Policy Gradient methods vs DQN

We'll start by discussing the general class of **policy gradient methods** (of which PPO is a member), and compare it to DQN. To recap, DQN works as follows:

> DQN involved learning the Q-function $Q(s, a)$, which represents the expected time-discounted future reward of taking action $a$ in state $s$. The update steps were based on the Bellman equation - this equation is only satisfied if we've found the true Q-function, so we minimize the squared TD residual to find it. We can derive the optimal policy by argmaxing $Q(s, a)$ over actions $a$.

On the other hand, policy gradient methods takes a more direct route - we write the expected future reward $J(\pi_\theta)$ as a function of our policy $\pi_\theta(a_t \mid s_t)$ (which takes the form of a neural network mapping from states to action logits), and then perform gradient ascent on this function to improve our policy directly i.e. $\theta \leftarrow \theta + \alpha \nabla_\theta J(\pi_\theta)$. In this way, we essentially sidestep having to think about the Bellman equation at all, and we directly optimize our policy function.

A question remains here - how can we take the derivative of expected future returns as a function of our policy $\pi_\theta(a_t \mid s_t)$, in a way which is differentiable wrt $\theta$? We'll discuss this in the next section.

> #### Summary so far
> 
> - In **policy gradient methods**, we directly optimize the policy function $\pi_\theta(a_t \mid s_t)$ to get higher expected future rewards.

<!-- 
|                     | DQN                                                                                               | PPO                                                                                                                                                                                                                                                                                                                                                                                                          |
|---------------------|--------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **What do we learn?** | We learn the Q-function $Q(s, a)$.                                                         | We learn the policy function $\pi(a \mid s)$.                                                                                                                                                                                                                                                                                                                     |
| **Where do our actions come from?** | Argmaxing $Q(s, a)$ over actions $a$ gives us a deterministic policy. We combine this with an epsilon-greedy algorithm when sampling actions, to enable exploration. | We directly learn our stochastic policy $\pi$, and we can sample actions from it.                                                                                                                                                                                                                                           |
| **What networks do we have?** | Our network `q_network` takes $s$ as inputs and outputs the Q-values for each possible action $a$. We also had a `target_network`, although this was just a lagged version of `q_network` rather than one that actually gets trained. | We have two networks: `actor`, which learns the policy function, and `critic`, which learns the value function $V(s)$. These two work in tandem: the `actor` requires the `critic`'s output to estimate the policy gradient and perform gradient ascent, and the `critic` tries to learn the value function of the `actor`'s current policy.                        |
| **Where do our gradients come from?** | We do gradient descent on the squared TD residual, i.e. the residual of the Bellman equation (which is only satisfied if we've found the true Q-function). | For our `actor`, we do gradient ascent on an estimate of the time-discounted future reward stream (i.e. we're directly moving up the **policy gradient**; changing our policy in a way that leads to higher expected reward). Our `critic` trains by minimizing the TD residual.                                                                                   |
| **Techniques to improve stability?** | We use a "lagged copy" of our network to sample actions from; in this way, we don't update too fast after only having seen a small number of possible states. In the DQN code, this was `q_network` and `target_network`. | We use a "lagged copy" of our policy in mathematical notation, this is $\theta$ and $\theta_{old}$. In the code, we won't actually need to make a different network for this. We clip the objective function to make sure large policy changes aren't incentivized past a certain point (this is the "proximal" part of PPO).                                     |
| **Suitable for continuous action spaces?** | No. Our Q-function $Q$ is implemented as a network which takes in states and returns Q-values for each discrete action. It's not even good for large action spaces! | Yes. Our policy function $\pi$ can take continuous argument $a$.                                                                                                                                                                                                                                                       |
-->

## Policy Gradient objective function

> Note, this is the most mathematically dense part of the material, and probably the part that's most fine to skip.

Let $\pi_\theta$ be our policy parameterized by $\theta$, and $J(\pi_\theta)$ denote the expected return of the policy (assuming undiscounted & finite-horizon for simplicity). Let $J(\pi_\theta)$ be the expected return of the policy $\pi_\theta$:
$$
J(\pi_\theta) = \underset{\tau \sim \pi_\theta}{\mathbb{E}} \left[ \sum_{t=0}^T r_{t+1}(s_t, a_t, s_{t+1}) \right]
$$

where $\tau = (s_0, a_0, ..., s_T)$ stands for a trajectory sampled from policy $\pi_\theta$, and we've written the rewards as $r_{t+1}(s_t, a_t, s_{t+1})$ to make it clear that they are a function of the trajectory. Then a theorem sometimes called the **policy gradient theorem** (PGT) says that the gradient of $J(\pi_\theta)$ is:
$$
\nabla_\theta J\left(\pi_\theta\right)=\underset{\tau \sim \pi_\theta}{\mathbb{E}}\left[\sum_{t=0}^T \nabla_\theta \log \pi_\theta\left(a_t \mid s_t\right) A_\theta(s_t, a_t)\right] \quad (*)
$$

where $A_\theta(s_t, a_t)$ is the **advantage function**, defined as $Q_\theta(s_t, a_t) - V_\theta(s_t)$, i.e. how much better it is to choose action $a_t$ in state $s_t$ as compared to the value obtained by following $\pi_\theta$ from that state onwards.

The derivation is optional (included in a dropdown below for completeness), but it is worth thinking about this expression intuitively:

- If the advantage $A_\theta(s_t, a_t)$ is positive, this means taking action $a_t$ in state $s_t$ is better on average than what we'd do under $\pi_\theta$. So if we increase $\pi_\theta(a_t \mid s_t)$ then $J(\pi_\theta)$ will increase, and vice-versa.
- If the advantage $A_\theta(s_t, a_t)$ is negative, this means taking action $a_t$ in state $s_t$ is worse than expectation, so if we increase $\pi_\theta(a_t \mid s_t)$ then $J(\pi_\theta)$ will decrease, and vice-versa.

So this expression is telling us that **we should change our policy $\pi_\theta(a_t \mid s_t)$ in a way which makes advantageous actions more likely, and non-advantageous actions less likely.** All pretty intuitive!

Note that instead of advantages $A_\theta(s_t, a_t)$, we could just use the total reward $R(\tau)$ in our objective function (we show this in the derivation below, because proving $(*)$ with $R(\tau)$ instead is actually an intermediate step in the proof). We call this algorithm REINFORCE. It can work, but the problem is it leads to much higher variance in our assessment of the value of actions. As a thought experiment: imagine you're assessing moves in a game of chess which you ended up winning. With $R(\tau)$, every move contributing to the win is rewarded equally, and so we can't differentiate between excellent moves and average ones. The advantage function is a lot more fine-grained: it allows you to answer the question "was this move better than what I typically play in this position?" for each move, isolating the contribution of the move from the game outcome as a whole.

<details>
<summary>Full derivation (optional)</summary>

> Summary: we can quite easily get a formula for $\nabla_\theta J\left(\pi_\theta\right)$ which looks like the one above, but has the total reward $R(\tau)$ instead of the advantage $A_\theta(s_t, a_t)$. We can then use a bag of tricks to show that we can subtract baseline functions from $R(\tau)$ in our expression without changing its value, until we get to an expression with $A_\theta(s_t, a_t)$ instead.

Let's go through the derivation line by line. Denoting $\int_\tau$ as an integral over the distribution of all states & actions in the trajectory, we have:

$$
\begin{aligned}
\nabla_\theta J\left(\pi_\theta\right) & =\nabla_\theta \underset{\tau \sim \pi_\theta}{\mathbb{E}}[R(\tau)] \\
& =\nabla_\theta \int_\tau P(\tau \mid \theta) R(\tau) \quad \text{Expand integration} \\
& =\int_\tau \nabla_\theta P(\tau \mid \theta) R(\tau) \quad \text{Bring gradient inside integral} \\
& =\int_\tau P(\tau \mid \theta) \nabla_\theta \log P(\tau \mid \theta) R(\tau) \quad \text{Use log derivative trick} \\
& =\underset{\tau \sim \pi_\theta}{\mathbb{E}}\left[\nabla_\theta \log P(\tau \mid \theta) R(\tau)\right] \quad \text{Recognize expectation} \\
& =\underset{\tau \sim \pi_\theta}{\mathbb{E}}\left[\sum_{t=0}^T \nabla_\theta \log \pi_\theta\left(a_t \mid s_t\right) R(\tau)\right] \quad (*)
\end{aligned}
$$

Where the log derivative trick is a rearrangement of the standard result $\nabla_\theta \log P_\theta(x) = \frac{\nabla_\theta P_\theta(x)}{P_\theta(x)}$, and the final line was reached by writing $P(\tau \mid \theta)$ as a product of transition probabilities and policy probabilities $\pi_\theta(a_t \mid s_t)$, using the fact that log of a product is the sum of logs, meaning we get:
$$
\log P(\tau \mid \theta) = \sum_{t=0}^T \log \pi_\theta(a_t \mid s_t) + \sum_{t=0}^T \log P(s_{t+1} \mid s_t, a_t)
$$
and the latter term vanishes when we take the gradient wrt $\theta$ because it's independent of $\theta$.

This formula looks like the one we had above, the only difference is that we have the total trajectory reward $R(\tau)$ instead of the advantage $A_\theta(s_t, a_t)$. And intuitively it seems like we should be able to get to this - after all, $R(\tau)$ is the sum of rewards at each timestep, and rewards accumulated before time $t$ shouldn't affect whether $\pi_\theta(a_t \mid s_t)$ should be increased or decreased. Although this is very intuitive, it turns out we have to do a bit of work to prove it.

Firstly, let's establish a lemma called the **expected grad-log-prob lemma**. This states that the expected gradient of a probability distribution is always zero:

$$
\underset{x \sim P_\theta}{\mathbb{E}}\left[\nabla_\theta \log P_\theta(x)\right] = 0
$$

Proof: we can write the expectation above as the integral $\int_x P_\theta(x) \nabla_\theta \log P_\theta(x) dx$, which can be written as $\int_x \nabla_\theta P_\theta(x) dx$ by the log-derivative trick. Then we swap the order of integration and differentiation to get $\nabla_\theta \int_x P_\theta(x) dx$, and then using the fact that $P_\theta$ is a probability distribution, this becomes $\nabla_\theta 1 = 0$.

To return to our policy gradient setting, not only does this show us that $\mathbb{E}_{\tau \sim \pi_\theta}\left[\nabla_\theta \log \pi_\theta(a_t \mid s_t)\right] = 0$, it also shows us that $\mathbb{E}_{\tau \sim \pi_\theta}\left[\nabla_\theta \log \pi_\theta(a_t \mid s_t) f(\tau) \right] = 0$ whenever the function $f(\tau)$ is only a function of the early trajectory values $s_0, a_0, ..., s_{t-1}, a_{t-1}, s_t$, because this still falls out as zero when we integrate over the distribution of our action $a_t$. This means that in $(*)$, we can actually replace $R(\tau)$ with $R(\tau) - f(\tau)$ for any such choice of $f(\tau)$. We choose $f(\tau) = \mathbb{E}_{\tau \sim \pi_\theta}\left[R(\tau) \mid s_t\right]$, i.e. the expected return conditioned on the trajectory up to  the early trajectory values. The already-accumulated rewards $r_1, ..., r_t$ cancel, and so in $(*)$ the term for any given timestep $t$ becomes:

$$
\underset{s_0, ..., s_t, a_t}{\mathbb{E}}\left[\nabla_\theta \log \pi_\theta\left(a_t \mid s_t\right) \left( \underset{\tau \sim \pi_\theta}{\mathbb{E}}\left[R(\tau) \mid s_0, ..., s_t, a_t\right] - \underset{\tau \sim \pi_\theta}{\mathbb{E}}\left[R(\tau) \mid s_0, ..., s_t\right] \right) \right]
$$

but since the first of these terms conditions on the action $a_t$ and the second doesn't, we recognize the term in the large brackets as exactly $Q_\theta(s_t, a_t) - V_\theta(s_t) = A_\theta(s_t, a_t)$, as required.

</details>

We have this formula, but how do we use it to get an objective function we can optimize for? The answer is that we take estimates of the advantage function $\hat{A}_{\theta_\text{target}}(s_t, a_t)$ using a frozen version of our parameters $\theta_{\text{target}}$ (like we took next-step Q-values from a frozen target network in DQN), and use our non-frozen parameters to get our values $\pi_\theta(s_t \mid a_t)$. For a given batch of experiences $B$ (which can be assumed to be randomly sampled across various different trajectories $\tau$), our objective function is:
$$
L(\theta) = \frac{1}{|B|} \sum_{t \in B} \log \pi_\theta(a_t \mid s_t) \hat{A}_{\theta_\text{target}}(s_t, a_t) 
$$
because then:
$$
\nabla_\theta L(\theta) = \frac{1}{|B|} \sum_{t \in B} \nabla_\theta \log \pi_\theta(a_t \mid s_t) \hat{A}_{\theta_\text{target}}(s_t, a_t) \approx \nabla_\theta J(\pi_\theta)
$$
exactly as we want! We can now perform gradient ascent on this objective function to improve our policy: $\theta \leftarrow \theta + \alpha \nabla_\theta L(\theta)$ will be an approximation of the ideal update rule $\theta \leftarrow \theta + \alpha \nabla_\theta J(\pi_\theta)$.

> #### Summary so far
> 
> - In **policy gradient methods**, we directly optimize the policy function $\pi_\theta(a_t \mid s_t)$ to get higher expected future rewards.
> - Our objective function is a sum of logprobs of actions taken, weighted by their advantage estimates $\hat{A}_\theta(s_t, a_t)$ (i.e. how good we think that action was), so performing gradient ascent on this leads to taking more good actions and less bad ones.
>   - Note that we could just use accumulated rewards $R(\tau)$ rather than the advantage function, but using advantage estimates is a lot more stable.

<details>
<summary>Question - can you intuitively explain how the advantage function influences policy updates?</summary>

The advantage function scales updates; positive $A_\theta$ will cause us to increase the action likelihood (becasuse the probability of that action will have a positive coefficient in the objective function), and negative $A_\theta$ will cause us to decrease the action likelihood.

</details>

## Actor & critic

Unlike DQN, we require 2 different networks for policy gradient methods:

- `actor`: learns the policy function $\pi_\theta(a_t \mid s_t)$, i.e. inputs are $s_t$ and outputs (for discrete action spaces) are a vector of logits for each action $a_t$
- `critic`: learns the value function $V_\theta(s_t)$ which is used to estimate the advantage function estimates $\hat{A}_\theta(s_t, a_t)$, i.e. inputs are $s_t$ and outputs a single scalar value

As we discussed in the last section, estimating $\hat{A}_\theta(s_t, a_t)$ is valuable because without it we'd have to rely on the accumulated reward $R(\tau)$ in our objective function, which is very high-variance and doesn't allow for granular credit assignment to different actions. In simple environments like CartPole you may find you can get by without the critic, but as we move into more complex environments this ceases to be the case.

Note - unlike DQN, **policy gradient methods are able to take continuous action spaces**. This is because we can have our `actor` output a vector of means and variances parameterising a distribution over actions, and then sample actions from this distribution. On the other hand, our Q-network in DQN is only able to take in states and output a single scalar value for each discrete action. This will be important when we look at MuJoCo later.

You might have a question at this point - **how does the critic learn the value function**? After all, the loss function $L(\theta)$ is designed just to update the policy $\pi_\theta$ (i.e. the actor network), not the critic network. The critic is used to compute the advantage function estimates, but these come from $\theta_\text{old}$ in the objective function $L(\theta)$, i.e. they don't track gradients. The answer is that we improve our value function estimates by adding another loss term which **minimizes the TD residual**. We'll add to the term $(V_\theta(s_t) - V_t^\text{target})^2$ into our loss function, where $V_\theta(s_t)$ are the value function estimates from our critic network (which do track gradients) and $V_t^\text{target} := V_{\theta_\text{target}}(s_t) + \hat{A}_{\theta_\text{target}}(s_t, a_t)$ are the next-step value estimates taken from our target network, which take into account the actual action taken $a_t$ and how much it changed our value.

> #### Summary so far
> 
> - In **policy gradient methods**, we directly optimize the policy function $\pi_\theta(a_t \mid s_t)$ to get higher expected future rewards.
> - Our objective function is a sum of logprobs of actions taken, weighted by their advantage estimates $\hat{A}_\theta(s_t, a_t)$ (i.e. how good we think that action was), so performing gradient ascent on this leads to taking more good actions and less bad ones.
>   - Note that we could just use accumulated rewards $R(\tau)$ rather than the advantage function, but using advantage estimates is a lot more stable.
> - We have 2 networks: `actor` which learns $\pi_\theta(a_t \mid s_t)$ using this objective function, and `critic` which learns $V_\theta(s_t)$ by minimizing the TD residual (a bit like SARSA), and allows us to estimate the advantage $\hat{A}_\theta(s_t, a_t)$ which is used in the objective function.

<details>
<summary>Question - why do policy gradient methods require both actor and critic networks, and how do they complement each other?</summary>

The actor learns the policy; the critic estimates value functions for stable advantage calculation. Without the actor we wouldn't have any policy to learn the value for, and without the critic we wouldn't be able to competently estimate the advantage function which is necessary so that we can compute our objective function / understand how we should update our policy.

</details>

<details>
<summary>Question - why is the critic network's loss function conceptually similar to the update rule we used for SARSA?</summary>

The SARSA update rule was:

$$
Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \eta \left( r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t,a_t) \right)
$$

which is actually equivalent to the update rule we'd get if our loss function was the squared TD error to our $Q$ function. Our critic loss function is pretty much the same idea, except we're applying the TD error to $V(s_t)$ rather than $Q(s_t, a_t)$.

Note that SARSA differed from Q-Learning/DQN because the latter also included a maximization over the action space - we were essentially doing policy improvement and learning the value function for it at the same time. Here, our critic loss function is more conceptually similar to SARSA than it is to Q-Learning/DQN, because the policy improvement is coming from the actor network instead.


</details>

## Generalized Advantage Estimation

We've got a lot of the pieces in place for PPO now - we have an actor and critic network, and we have 2 objective functions: one to train the critic to estimate the value function $V_\theta(s_t)$ accurately (which are used to estimate the advantage function $\hat{A}_\theta(s_t, a_t)$), and one which trains the actor to maximize the expected future reward based on these advantage estimates. 

A question remains now - how do we use value estimates to compute advantage estimates? Here are some ideas:

1. We can use the 1-step residual, i.e. $\hat{A}_\theta(s_t, a_t) = \delta_t = r_t + \gamma V_\theta(s_{t+1}) - V_\theta(s_t)$ just like we used in DQN. The problem with this is that we're only estimating the advantage based on a single action taken, which is a bit too myopic. If we sacrifice a piece in our chess game to win the match, we want to make sure our advantage estimates take this future position into account, rather than just thinking that we're down one piece!
2. We can use the sum of future residuals, i.e. $\hat{A}_\theta(s_t, a_t) = \delta_t + \gamma \delta_{t+1} + ...$. This fixes the myopia problem, but brings back a new problem - doing this is pretty much just like using $R(\tau)$ in our objective function instead, in other words we're looking at the entire future trajectory at once! This leads to unstable training, and an inability to credit any individual action.

The solution is a middleground between these two: we perform **generalized advantage estimation** (GAE), which is a sum of future residuals but geometrically decaying according to some factor $\lambda$. In other words, we take $\hat{A}^{\text{GAE}(\lambda)}_\theta(s_t, a_t) = \delta_t + \lambda \gamma \delta_{t+1} + \lambda^2 \gamma^2 \delta_{t+2} + ...$. This is effectively the best of both worlds - we put the largest weight on the next actions taken (allowing us to attribute actions rather than entire trajectories), but also we do take into account future states in our trajectory (meaning we're not only concerned with the immediate reward). Note that $\lambda=0$ reduces to the first idea, and $\lambda=1$ to the second.

Note that the fact that we use GAE also helps a lot for our critic network - in SARSA we were minimizing the 1-step TD error, but here we're training $V(s_t)$ to be more in line with a lookahead estimate of the value function which takes into account many future actions and states. This helps improve stability and speed up convergence.

> #### Summary so far
> 
> - In **policy gradient methods**, we directly optimize the policy function $\pi_\theta(a_t \mid s_t)$ to get higher expected future rewards.
> - Our objective function is a sum of logprobs of actions taken, weighted by their advantage estimates $\hat{A}_\theta(s_t, a_t)$ (i.e. how good we think that action was), so performing gradient ascent on this leads to taking more good actions and less bad ones.
>   - Note that we could just use accumulated rewards $R(\tau)$ rather than the advantage function, but using advantage estimates is a lot more stable.
> - We have 2 networks: `actor` which learns $\pi_\theta(a_t \mid s_t)$ using this objective function, and `critic` which learns $V_\theta(s_t)$ by minimizing the TD residual (a bit like SARSA), and allows us to estimate the advantage $\hat{A}_\theta(s_t, a_t)$ which is used in the objective function.
> - We use **generalized advantage estimation** (GAE) to convert our value function estimates into advantage estimates - this mostly avoids the two possible extremes of (1) myopia from only looking at the immediate reward, and (2) instability / failure to credit single actions from looking at the entire trajectory at once.

<details>
<summary>Question - can you explain why using GAE is better than using the realized return trajectory in our loss function?</summary>

GAE is much more stable, because using the entire trajectory means we're only taking into account the actual reward accumulated (which can have much higher variance than an advantage estimate, assuming we already have a good policy). Additionally, GAE allows us to credit individual actions for the future rewards they lead to, which is something we couldn't do with the realized return trajectory.

</details>

## PPO

We're pretty much there now - we've established all the theoretical pieces we need for PPO, and there's just 3 final things we need to add to the picture.

1. **We use an entropy bonus to encourage policy exploration, and prevent premature convergence to suboptimal policies.**

Entropy is a very deep mathematical topic that we won't dive all the way into here, but for the sake of brevity, we can say that entropy is a measure of uncertainty - policies which will definitely take the same action in the same state have low entropy, and policies which have a wide range of likely actions have high entropy. We add some multiple of the entropy of our policy function directly onto our objective function to be maximized, with the entropy coefficient usually decaying over time as we move from explore to exploit mode.

2. **We clip the objective function $L(\theta)$ to get $L^\text{CLIP}(\theta)$, to prevent the policy from changing too much too fast.**

The clipping is applied to make sure the ratio $\pi_\theta(a_t \mid s_t) / \pi_{\theta_\text{target}}(a_t \mid s_t)$ stays close to 1, during any single learning phase (between learning phases we generate a new batch of experiences and reset $\theta_\text{target}$). Intuitively, this is because the more our policy changes from the old policy, the more unrealistic the generated experiences will be. For example, suppose we generate experiences from a bunch of chess games, where a particular class of strategies (e.g. playing more aggressively) is beneficial. We shouldn't update on these games indefinitely, because as we update and the agent's policy changes to become more aggressive, the generated experiences will no longer be accurate representations of the agent's strategy and so our objective function will no longer be a good estimate of the expected future reward.

There are various ways to keep this ratio close to 1. **Trust region policy optimization** (TRPO) explicitly adds a multiple of the [KL divergence](https://www.lesswrong.com/posts/no5jDTut5Byjqb4j5/six-and-a-half-intuitions-for-kl-divergence) to the loss function, making sure the distributions stay close. PPO however does something a lot more simple and hacky - if the ratio is larger than $1+\epsilon$ for actions with positive advantage (for some constant $\epsilon > 0$) then we clip it, preventing gradients from flowing & updating the policy network more. We do the same thing with $1-\epsilon$ when the advantage is negative.

3. **We actually use $\dfrac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_\text{target}}(a_t \mid s_t)}$ rather than $\log \pi_\theta(a_t \mid s_t)$ when computing $L^\text{CLIP}(\theta)$.**

This is a valid thing to do precisely because we're using clipping - this allows us to assume the probability ratio is usually close to 1, and so we can use the approximation $\log(x) \approx x - 1$ for all $x \approx 1$ (with this approximation, the two loss functions are equal up to a constant that doesn't depend on $\theta$ - we leave the proof as an exercise to the reader).

Although these 3 changes are all important, it's the 2nd alteration that distinguishes PPO from other policy gradient methods. It's where it gets its name - **proximal** refers to the way in which this clipping keeps us close to the old policy.

> #### Full summary
> 
> - In **policy gradient methods**, we directly optimize the policy function $\pi_\theta(a_t \mid s_t)$ to get higher expected future rewards.
> - Our objective function is a sum of logprobs of actions taken, weighted by their advantage estimates $\hat{A}_\theta(s_t, a_t)$ (i.e. how good we think that action was), so performing gradient ascent on this leads to taking more good actions and less bad ones.
>   - Note that we could just use accumulated rewards $R(\tau)$ rather than the advantage function, but using advantage estimates is a lot more stable.
> - We have 2 networks: `actor` which learns $\pi_\theta(a_t \mid s_t)$ using this objective function, and `critic` which learns $V_\theta(s_t)$ by minimizing the TD residual (a bit like SARSA), and allows us to estimate the advantage $\hat{A}_\theta(s_t, a_t)$ which is used in the objective function.
> - We use **generalized advantage estimation** (GAE) to convert our value function estimates into advantage estimates - this mostly avoids the two possible extremes of (1) myopia from only looking at the immediate reward, and (2) instability / failure to credit single actions from looking at the entire trajectory at once.
> - On top of all this, 2 other techniques fully characterize PPO:
>   - We add an **entropy bonus** to our objective function to encourage exploration.
>   - We clip the objective function so $\pi_\theta(a_t \mid s_t)$ isn't incentivized to change too quickly (which could cause instability) - this is the "proximal" part of PPO.

<details>
<summary>Question - what is the role of the entropy term, and should it be added to or subtracted from the clipped objective function?</summary>

The entropy term encourages policy exploration. We want to add it to the objective function, because we're doing gradient ascent on it (and we want to increase exploration).

</details>

<details>
<summary>Question - how does clipping the objective function help prevent large policy updates, and why is this desireable?</summary>

Clipping prevents large policy changes by capping gradients when the update exceeds some value $\epsilon$, generally ensuring proximity to the old policy.

This is good because we don't want to change our policy too quickly, based on possibly a limited set of experiences.

</details>

## Today's setup

We've now covered all the theory we need to understand about PPO! To conclude, we'll briefly go through how our PPO algorithm is going to be set up in practice today, and relate it to what we've discussed in the previous sections.

A full diagram of our implementation is shown below:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ppo-alg-conceptual.png" width="900">

We have 2 main phases: the **rollout phase** (where we generate a batch of experiences from our frozen network $\theta_\text{target}$) and the **learning phase** (where we update our policy $\pi_\theta$ based on the experiences generated in the rollout phase, as well as the outputs of our current network $\theta$). This is quite similar to the setup we used for DQN (where we'd alternate between generating experiences for our buffer and learning from those experiences) - the difference here is that rather than the rollout phase adding to our buffer, it'll be emptying our buffer and creating an entirely new one. So as not to be wasteful, the learning phase will iterate over our buffer multiple times before we repeat the cycle.

Just like we had `ReplayBuffer`, `DQNAgent` and `DQNTrainer` as our 3 main classes yesterday, here we have 3 main classes:

- `ReplayMemory` stores experiences generated by the agent during rollout, and has a `get_minibatches` method which samples data to be used in the learning phase
- `PPOAgent` manages the interaction between our policy and environment (particularly via the `play_step` method), it generates experiences and adds them to our memory
- `PPOTrainer` is the main class for training our model, it's essentially a wrapper around everything else

As we can see in the diagram, the learning phase has us compute an objective function which involves 3 different terms:

- The **entropy bonus** (for encouraging policy exploration) - this trains only our actor network
- The **clipped surrogate objective function** (for policy improvement) - this trains only our actor network (although it uses the critic network's estimates from $\theta_\text{target}$), and it's the most important of the three terms
- The **value function loss** (for improving our value function estimates) - this trains only our critic network

# 1️⃣ Setting up our agent

> ##### Learning Objectives
>
> - Understand the difference between the actor & critic networks, and what their roles are
> - Learn about & implement generalised advantage estimation
> - Build a replay memory to store & sample experiences
> - Design an agent class to step through the environment & record experiences

In this section, we'll do the following:

* Define a dataclass to hold our PPO arguments
* Write functions to create our actor and critic networks (which will eventually be stored in our `PPOAgent` instance)
* Write a function to do **generalized advantage estimation** (this will be necessary when computing our objective function during the learning phase)
* Fill in our `ReplayMemory` class (for storing and sampling experiences)
* Fill in our `PPOAgent` class (a wrapper around our networks and our replay memory, which will turn them into an agent)

As a reminder, we'll be continually referring back to [The 37 Implementation Details of Proximal Policy Optimization](https://iclr-blog-track.github.io/2022/03/25/ppo-implementation-details/#solving-pong-in-5-minutes-with-ppo--envpool) as we go through these exercises. Most of our sections wil refer to one or more of these details.

## PPO Arguments

Just like for DQN, we've provided you with a dataclass containing arguments for your `train_ppo` function. We've also given you a function from `utils` to display all these arguments (including which ones you've changed). Lots of these are the same as for the DQN dataclass.

Don't worry if these don't all make sense right now, they will by the end.

In [ ]:
@dataclass
class PPOArgs:
    # Basic / global
    seed: int = 1
    env_id: str = "CartPole-v1"
    mode: Literal["classic-control", "atari", "mujoco"] = "classic-control"

    # Wandb / logging
    use_wandb: bool = False
    video_log_freq: int | None = None
    wandb_project_name: str = "PPOCartPole"
    wandb_entity: str = None

    # Duration of different phases
    total_timesteps: int = 500_000
    num_envs: int = 4
    num_steps_per_rollout: int = 128
    num_minibatches: int = 4
    batches_per_learning_phase: int = 4

    # Optimization hyperparameters
    lr: float = 2.5e-4
    max_grad_norm: float = 0.5

    # RL hyperparameters
    gamma: float = 0.99

    # PPO-specific hyperparameters
    gae_lambda: float = 0.95
    clip_coef: float = 0.2
    ent_coef: float = 0.01
    vf_coef: float = 0.25

    def __post_init__(self):
        self.batch_size = self.num_steps_per_rollout * self.num_envs

        assert self.batch_size % self.num_minibatches == 0, "batch_size must be divisible by num_minibatches"
        self.minibatch_size = self.batch_size // self.num_minibatches
        self.total_phases = self.total_timesteps // self.batch_size
        self.total_training_steps = self.total_phases * self.batches_per_learning_phase * self.num_minibatches

        self.video_save_path = section_dir / "videos"


args = PPOArgs(num_minibatches=2)  # changing this also changes minibatch_size and total_training_steps
arg_help(args)

A note on the `num_envs` argument - note that unlike yesterday, `envs` will actually have multiple instances of the environment inside (we did still have this argument yesterday but it was always set to 1). From the [37 implementation details of PPO](https://iclr-blog-track.github.io/2022/03/25/ppo-implementation-details/#:~:text=vectorized%20architecture) post:

> _In this architecture, PPO first initializes a vectorized environment `envs` that runs $N$ (usually independent) environments either sequentially or in parallel by leveraging multi-processes. `envs` presents a synchronous interface that always outputs a batch of $N$ observations from $N$ environments, and it takes a batch of $N$ actions to step the $N$ environments. When calling `next_obs = envs.reset()`, next_obs gets a batch of $N$ initial observations (pronounced "next observation"). PPO also initializes an environment `done` flag variable next_done (pronounced "next done") to an $N$-length array of zeros, where its i-th element `next_done[i]` has values of 0 or 1 which corresponds to the $i$-th sub-environment being *not done* and *done*, respectively._

## Actor-Critic Implementation ([detail #2](https://iclr-blog-track.github.io/2022/03/25/ppo-implementation-details/#:~:text=Orthogonal%20Initialization%20of%20Weights%20and%20Constant%20Initialization%20of%20biases))

PPO requires two networks, an `actor` and a `critic`. The actor is the most important one; its job is to learn an optimal policy $\pi_\theta(a_t \mid s_t)$ (it does this by training on the clipped surrogate objective function, which is essentially a direct estimation of the discounted sum of future rewards with some extra bells and whistles thrown in). Estimating this also requires estimating the **advantage function** $A_\theta(s_t, a_t)$, which in requires estimating the values $V_\theta(s_t)$ - this is why we need a critic network, which learns $V_\theta(s_t)$ by minimizing the TD residual (in a similar way to how our Q-network learned the $Q(s_t, a_t)$ values).

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "2.3.10"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part3_ppo.solutions import layer_init, compute_advantages, get_minibatch_indices, PPOAgent, calc_clipped_surrogate_objective, calc_value_function_loss, calc_entropy_bonus, make_optimizer, PPOTrainer, EasyCart


### Exercise - rewrite `get_actor_and_critic`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

The function `get_actor_and_critic` had a boolean argument `atari`, which we ignored previously, but which we'll now return to. When this argument is `False` then the function should behave exactly as it did before (i.e. the Cartpole version), but when `True` then it should return a shared CNN architecture for the actor and critic. The architecture should be as follows (you can open it in a new tab if it's hard to see clearly):

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/mermaid-diagram-2024-11-28-154133.svg" style="background-color: #fffbe0; padding: 10px;" width="350" height="1100">

<!-- 
flowchart TD
    A["Input<br>shape = (4, L, L)"] -> B["8x8 Conv<br>32 Channels<br>Padding 0<br>Stride 4"]
    B -> C["ReLU"]
    C -> D["4x4 Conv<br>64 Channels<br>Padding 0<br>Stride 2"]
    D -> E["ReLU"]
    E -> F["3x3 Conv<br>64 Channels<br>Padding 0<br>Stride 1"]
    F -> G[Flatten]
    G -> H[ReLU]
    H -> I["Linear<br>512 outputs"]
    I -> J["ReLU"]
    J -> K1["Linear(512, n_act)<br>row_norm=0.01"]
    K1 -> L1["Actor output"]
    J -> K2["Linear(512, 1)<br>row_norm=1"]
    K2 -> L2["Critic output"]

{
  "theme": "default",
  "themeVariables": {
    "fontSize": "22px"
    }
}
-->


Note - when calculating the number of input features for the linear layer, you can assume that the value `L` is 4 modulo 8, i.e. we can write `L = 8m + 4` for some integer `m`. This will make the convolutions easier to track. You shouldn't hardcode the number of input features assuming an input shape of `(4, 84, 84)`; this is bad practice!

We leave the exercise of finding the number of input features to the linear layer as a challenge for you. If you're stuck, you can find a hint in the section below (this isn't a particularly conceptually important detail).

<details>
<summary>Help - I don't know what the number of inputs for the first linear layer should be.</summary>

You can test this empirically by just doing a forward pass through the first half of the network and seeing what the shape of the output is.

Alternatively, you can use the convolution formula. There's never any padding, so for a conv with parameters `(size, stride)`, the dimensions change as `L -> 1 + (L - size) // stride` (see the [documentation page](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)). So we have:

```
8m+4 -> 1 + (8m-4)//4 = 2m
2m   -> 1 + (2m-4)//2 = m-1
m-1  -> 1 + (m-4)//1  = m-3
```

For instance, if `L = 84` then `m = 10` and `L_new = m-3 = 7`. So the linear layer is fed 64 features of shape `(64, 7, 7)`.

</details>

Now, you can fill in the `get_actor_and_critic_atari` function below, which is called when we call `get_actor_and_critic` with `mode == "atari"`.

Note that we take the observation shape as argument, not the number of observations. It should be `(4, L, L)` as indicated by the diagram. The shape `(4, L, L)` is a reflection of the fact that we're using 4 frames of history per input (which helps the model calculate things like velocity), and each of these frames is a monochrome resized square image.

In [ ]:
def get_actor_and_critic_atari(obs_shape: tuple[int,], num_actions: int) -> tuple[nn.Sequential, nn.Sequential]:
    """
    Returns (actor, critic) in the "atari" case, according to diagram above.
    """
    assert obs_shape[-1] % 8 == 4

    raise NotImplementedError()


tests.test_get_actor_and_critic(get_actor_and_critic, mode="atari")

<details><summary>Solution</summary>

```python
def get_actor_and_critic_atari(obs_shape: tuple[int,], num_actions: int) -> tuple[nn.Sequential, nn.Sequential]:
    """
    Returns (actor, critic) in the "atari" case, according to diagram above.
    """
    assert obs_shape[-1] % 8 == 4

    L_after_convolutions = (obs_shape[-1] // 8) - 3
    in_features = 64 * L_after_convolutions * L_after_convolutions

    hidden = nn.Sequential(
        layer_init(nn.Conv2d(4, 32, 8, stride=4, padding=0)),
        nn.ReLU(),
        layer_init(nn.Conv2d(32, 64, 4, stride=2, padding=0)),
        nn.ReLU(),
        layer_init(nn.Conv2d(64, 64, 3, stride=1, padding=0)),
        nn.ReLU(),
        nn.Flatten(),
        layer_init(nn.Linear(in_features, 512)),
        nn.ReLU(),
    )

    actor = nn.Sequential(hidden, layer_init(nn.Linear(512, num_actions), std=0.01))
    critic = nn.Sequential(hidden, layer_init(nn.Linear(512, 1), std=1))

    return actor, critic
```
</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# Wrap tests.test_get_actor_and_critic so a passing test fires the beacon.
try:
    _dd_orig = tests.test_get_actor_and_critic
    def _dd_wrapped(*args, **kwargs):
        result = _dd_orig(*args, **kwargs)
        _dd_report_complete()
        return result
    tests.test_get_actor_and_critic = _dd_wrapped
except AttributeError:
    print('[Delta Drills] no matching test function — call _dd_report_complete() manually when done.')
